In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("ggplot")
pd.set_option("display.max_columns", None)

In [28]:
# Load all datasets
users = pd.read_csv("raw_data/users.csv")
movies = pd.read_csv("raw_data/movies.csv")
watch = pd.read_csv("raw_data/watch_history.csv")
recommendations = pd.read_csv("raw_data/recommendation_logs.csv")
search = pd.read_csv("raw_data/search_logs.csv")
reviews = pd.read_csv("raw_data/reviews.csv")

## Removed Duplicates

In [29]:
users = users.drop_duplicates(
    subset="email",
    keep="first"
)

movies = movies.drop_duplicates()

watch = watch.drop_duplicates()

recommendations = recommendations.drop_duplicates()

search = search.drop_duplicates()

reviews = reviews.drop_duplicates()

In [30]:
users.duplicated().sum()

np.int64(0)

## Changed Data Types 
Changed data types of date and time columns from str to datetime.

In [31]:

users["subscription_start_date"] = pd.to_datetime(
    users["subscription_start_date"]
)

users["created_at"] = pd.to_datetime(
    users["created_at"]
)

# Watch Table
watch["watch_date"] = pd.to_datetime(
    watch["watch_date"]
)

# Search Table
search["search_date"] = pd.to_datetime(
    search["search_date"]
)

# Reviews Table
reviews["review_date"] = pd.to_datetime(
    reviews["review_date"]
)

In [32]:
users.dtypes

user_id                               str
email                                 str
first_name                            str
last_name                             str
age                               float64
gender                                str
country                               str
state_province                        str
city                                  str
subscription_plan                     str
subscription_start_date    datetime64[us]
is_active                            bool
monthly_spend                     float64
primary_device                        str
household_size                    float64
created_at                 datetime64[us]
dtype: object

## Handle Missing Values

### Users Table 
1. Used median to fill age values as it wont be deviated by outliers.
2. Filled monthly_spend with total median values based on subscription plans.

In [33]:
users = users[(users["age"] >= 1) & (users["age"] <= 100)]

users["age"] = (
    users.groupby("subscription_plan")["age"]
    .transform(lambda x: x.fillna(x.median()))
)

users["gender"] = users["gender"].fillna("Unknown")

users["monthly_spend"] = (
    users.groupby("subscription_plan")["monthly_spend"]
               .transform(lambda x: x.fillna(x.median()))
)

users["household_size"] = (
    users["household_size"]
        .fillna(users["household_size"].median())
)

In [34]:
users.isnull().sum()
# users["age"].min()

user_id                    0
email                      0
first_name                 0
last_name                  0
age                        0
gender                     0
country                    0
state_province             0
city                       0
subscription_plan          0
subscription_start_date    0
is_active                  0
monthly_spend              0
primary_device             0
household_size             0
created_at                 0
dtype: int64

### Recommendation Table

In [35]:
# 1. First choice: Fill by (strategy + algorithm_version) group median
recommendations["recommendation_score"] = recommendations.groupby(
    ["recommendation_type", "algorithm_version"]
)["recommendation_score"].transform(lambda x: x.fillna(x.median()))

# 2. Second choice: Fill remaining NaNs by strategy group median
recommendations["recommendation_score"] = recommendations.groupby(
    "recommendation_type"
)["recommendation_score"].transform(lambda x: x.fillna(x.median()))

# 3. Final choice: Fill any leftover NaNs with overall dataset median
recommendations["recommendation_score"] = recommendations[
    "recommendation_score"
].fillna(recommendations["recommendation_score"].median())

In [36]:
recommendations["recommendation_score"].isnull().sum()

np.int64(0)

### Reviews Table

In [37]:
reviews[["helpful_votes", "total_votes"]] = reviews[
    ["helpful_votes", "total_votes"]
].fillna(0)

reviews["sentiment_score"] = reviews.groupby("sentiment")["sentiment_score"].transform(
    lambda x: x.fillna(x.median())
)

In [38]:
reviews["sentiment_score"].isnull().sum()

np.int64(0)

## Export as CSV

In [39]:
users.to_csv("users_cleaned.csv", index=False)
movies.to_csv("movies_cleaned.csv", index=False)
watch.to_csv("watch_cleaned.csv", index=False)
recommendations.to_csv("recommendations_cleaned.csv", index=False)
search.to_csv("search_cleaned.csv", index=False)
reviews.to_csv("reviews_cleaned.csv", index=False)